In [1]:
import os
import base64
from email.message import EmailMessage
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import sys
from google.cloud import pubsub_v1
import json

In [2]:
SCOPES = ['https://mail.google.com/']

creds = None
if os.path.exists('api_key/token.json'):
    creds = Credentials.from_authorized_user_file(f'api_key/token.json', SCOPES)

if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())
    else:
        flow = InstalledAppFlow.from_client_secrets_file(f'api_key/client_secret.json', SCOPES)
        creds = flow.run_local_server(port=0)

    with open(f'api_key/token.json', 'w') as token:
        token.write(creds.to_json())

service = build('gmail', 'v1', credentials=creds)

In [11]:
# service.users().stop(userId='me').execute()

In [12]:
# 1. Register the Watch Request
request_body = {
    'labelIds': ['INBOX'],
    'topicName': 'projects/mail-service-504402/topics/gmail-notifications',
    'labelFilterBehavior': 'INCLUDE'
}

watch_response = service.users().watch(userId='me', body=request_body).execute()

print("Watch response:", watch_response)

Watch response: {'historyId': '4200', 'expiration': '1786415234341'}


In [ ]:
project_id = "mail-service-504402"
subscription_id = "gmail-notifications-sub"

subscriber = pubsub_v1.SubscriberClient()
subscription_path = subscriber.subscription_path(project_id, subscription_id)

def callback(message):
    print(f"Received notification!")

    # Extract historyId to fetch updated emails
    history_id = data.get('historyId')
    print(f"Latest History ID: {history_id}")
    
    # Acknowledge the message so it isn't resent
    message.ack()

streaming_pull_future = subscriber.subscribe(subscription_path, callback=callback)
print(f"Listening for messages on {subscription_path}...\n")

try:
    streaming_pull_future.result()
except KeyboardInterrupt:
    streaming_pull_future.cancel()

In [5]:
import json
import concurrent.futures
from google.cloud import pubsub_v1

project_id = "mail-service-504402"
subscription_id = "gmail-notifications-sub"

subscriber = pubsub_v1.SubscriberClient()
subscription_path = subscriber.subscription_path(project_id, subscription_id)

def callback(message):
    data = json.loads(message.data.decode('utf-8'))
    print(f"Latest History ID: {data.get('historyId')}")
    print("Payload:", data)
    message.ack()
    
    

streaming_pull_future = subscriber.subscribe(subscription_path, callback=callback)
print(f"Listening for messages on {subscription_path}...")

# Listen for 280 seconds (~4.5 minutes) then exit
TIMEOUT = 1 

try:
    streaming_pull_future.result(timeout=TIMEOUT)
except concurrent.futures.TimeoutError:
    # Shutdown gracefully so cron can start a fresh process next cycle
    streaming_pull_future.cancel()
    print("5-minute window finished. Exiting gracefully.")

/Users/ginoprasad/miniconda3/envs/google-api/lib/python3.11/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
I0803 19:35:50.591354 2249529 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0803 19:35:50.598134 2251002 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(98, generation: 1)
I0803 19:35:50.598241 2251002 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(92, generation: 1)


Listening for messages on projects/mail-service-504402/subscriptions/gmail-notifications-sub...
5-minute window finished. Exiting gracefully.


In [ ]:
print("Hi")